## What are we trying to detect?

For every PDF page, we want to answer:

Does this page contain useful extractable text?

             │
        ┌────┴────┐
        │         │
       YES        NO
        │         │
        ▼         ▼
     Normal PDF  OCR candidate

But we'll go one step further.

A page can contain some text and still be problematic. So we'll collect useful diagnostics rather than simply returning True/False.

## PDF Page Inspection
##  Detecting Scanned / Non-Text PDF Pages

The objective is to inspect every PDF page before deciding whether normal
text extraction is sufficient or OCR may be required.

### Create the PDF paths and check their existance in the provided path

In [ ]:
from pathlib import Path

text_pdf_path = Path(
    "../../data/raw/pdf/azure_event_hubs_knowledge.pdf"
)

scanned_pdf_path = Path(
    "../../data/raw/pdf/azure_event_hubs_scanned.pdf"
)

print("Text PDF:")
print(text_pdf_path.resolve())
print("Exists:", text_pdf_path.exists())

print()

print("Scanned PDF:")
print(scanned_pdf_path.resolve())
print("Exists:", scanned_pdf_path.exists())

### Load the Text PDF 

In [15]:
from langchain_community.document_loaders import PyMuPDFLoader

C:\Users\Siva Ponakala\AppData\Local\Temp\ipykernel_15808\3177898484.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader


In [16]:
# load the text PDF using PyMuPDFLoader
text_loader = PyMuPDFLoader(str(text_pdf_path))
text_documents = text_loader.load()

### Load the Scanned PDF

In [17]:
# Load the scanned PDF using PyMuPDFLoader
scanned_loader = PyMuPDFLoader(str(scanned_pdf_path))
scanned_documents = scanned_loader.load()

#### Create the inspection function

In [18]:
def analyze_pdf_pages(documents):
    """
    Analyze PDF pages and return page-level diagnostics.
    """

    results = []

    for page_number, document in enumerate(documents, start=1):

        content = document.page_content.strip()

        results.append({
            "page": page_number,
            "characters": len(content),
            "has_text": bool(content),
            "metadata": document.metadata
        })

    return results

In [19]:
# now call the function for both text and scanned documents
text_analysis = analyze_pdf_pages(text_documents)

scanned_analysis = analyze_pdf_pages(scanned_documents)

In [20]:
text_analysis

[{'page': 1,
  'characters': 562,
  'has_text': True,
  'metadata': {'producer': 'ReportLab PDF Library - (opensource)',
   'creator': '(unspecified)',
   'creationdate': '2026-08-14T18:13:57+00:00',
   'source': '..\\..\\data\\raw\\pdf\\azure_event_hubs_knowledge.pdf',
   'file_path': '..\\..\\data\\raw\\pdf\\azure_event_hubs_knowledge.pdf',
   'total_pages': 5,
   'format': 'PDF 1.4',
   'title': '(anonymous)',
   'author': '(anonymous)',
   'subject': '(unspecified)',
   'keywords': '',
   'moddate': '2026-08-14T18:13:57+00:00',
   'trapped': '',
   'modDate': "D:20260814181357+00'00'",
   'creationDate': "D:20260814181357+00'00'",
   'page': 0}},
 {'page': 2,
  'characters': 363,
  'has_text': True,
  'metadata': {'producer': 'ReportLab PDF Library - (opensource)',
   'creator': '(unspecified)',
   'creationdate': '2026-08-14T18:13:57+00:00',
   'source': '..\\..\\data\\raw\\pdf\\azure_event_hubs_knowledge.pdf',
   'file_path': '..\\..\\data\\raw\\pdf\\azure_event_hubs_knowledge.pd

In [21]:
scanned_analysis

[{'page': 1,
  'characters': 0,
  'has_text': False,
  'metadata': {'producer': 'ReportLab PDF Library - (opensource)',
   'creator': 'anonymous',
   'creationdate': '2026-08-15T04:56:07+00:00',
   'source': '..\\..\\data\\raw\\pdf\\azure_event_hubs_scanned.pdf',
   'file_path': '..\\..\\data\\raw\\pdf\\azure_event_hubs_scanned.pdf',
   'total_pages': 1,
   'format': 'PDF 1.3',
   'title': 'untitled',
   'author': 'anonymous',
   'subject': 'unspecified',
   'keywords': '',
   'moddate': '2026-08-15T04:56:07+00:00',
   'trapped': '',
   'modDate': "D:20260815045607+00'00'",
   'creationDate': "D:20260815045607+00'00'",
   'page': 0}}]

### Make the result easier to understand

In [ ]:
import pandas as pd

In [22]:
text_df = pd.DataFrame(text_analysis)
text_df

,page,characters,has_text,metadata
0,1,562,True,{'producer': 'ReportLab PDF Library - (opensou...
1,2,363,True,{'producer': 'ReportLab PDF Library - (opensou...
2,3,382,True,{'producer': 'ReportLab PDF Library - (opensou...
3,4,362,True,{'producer': 'ReportLab PDF Library - (opensou...
4,5,465,True,{'producer': 'ReportLab PDF Library - (opensou...


In [23]:
scanned_df = pd.DataFrame(scanned_analysis)

scanned_df

,page,characters,has_text,metadata
0,1,0,False,{'producer': 'ReportLab PDF Library - (opensou...


## This makes the difference very obvious

By this step we confirm that we have a scanned version of PDF